<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_LB_%2038.84877.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 98% 91.0M/92.7M [00:00<00:00, 207MB/s]
100% 92.7M/92.7M [00:00<00:00, 195MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [ ]:
!pip install dask-cuda==24.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.4/134.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.10.0
    Uninstalling dask-2024.10.0:
      Successfully uninstalled dask-2024.10.0


In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 577, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 577 (delta 116), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (577/577), 188.95 KiB | 2.55 MiB/s, done.
Resolving deltas: 100% (290/290), done.
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 2.12s
 Downloaded ucx-py-cu12
 Downloaded cuspatial-cu12
 Downloaded libucx-cu12
 Downloaded datashader
 Downloaded libcuspatial-cu12
 Downloaded cucim-cu12
 Downloaded scikit-image
 Downloaded raft-dask-cu12
 Downloaded cuvs-cu12
 Downloaded cuml-cu12
 Downloaded cugraph-cu12
Prepared 21 packages in 25.42s
Uninstalled 1 package in 1.81s
Installed 21 packages in 414ms
 + cucim-cu12==24.12.0
 + cugraph-cu12==24.12.0
 + cuml-cu12==24.12.0
 + cuproj-cu12==24.12.0
 + cuspatial-cu12==24.12.0
 + cuvs-cu12==24.12.0
 + cuxfilter-cu

In [ ]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [ ]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [ ]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [ ]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)

In [ ]:
train = pd.concat([train,train_ex], axis = 0, ignore_index = True)

In [ ]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
oof = np.zeros(len(train))
preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    print(x_train.columns)

    model = CatBoostRegressor(
                              learning_rate = 0.11509572776170199,
                              l2_leaf_reg=5,
                              task_type = "GPU",
                              grow_policy = 'Lossguide',
                              random_state = 42,
                              cat_features = cats,
                              verbose = 250,
                              loss_function='RMSE')

    model.fit(x_train, y_train)

    val_preds = model.predict(x_val)

    oof[test_idx] = val_preds

    preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'weight_log',
       'size-laptop compartment_fe_TE', 'Color-waterproof_fe_TE',
       'weightcapacity-color_fe_TE', 'Brand_TE', 'Material_TE', 'Size_TE',
       'Compartments_TE', 'Laptop Compartment_TE', 'Waterproof_TE', 'Style_TE',
       'Color_TE', 'Weight Capacity (kg)_TE', 'colorxweight', 'Sizexweight',
       'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870475	total: 20.3ms	remaining: 20.3s
250:	learn: 38.6235020	total: 3.56s	remaining: 10.6s
500:	learn: 38.5951387	total: 6.79s	remaining: 6.76s
750:	learn: 38.5699464	total: 9.98s	remaining: 3.31s
999:	learn: 38.5467694	total: 13.3s	remaining: 0us
Fold: 1, Score: 38.639696120145665


In [ ]:
def objective(trial):

    params = {
        "learning_rate": 0.11509572776170199,
        "l2_leaf_reg": trial.suggest_int("l2_leaf_reg", 3, 15),
        #"subsample": trial.suggest_float("subsample", 0.05, 1.0),
        #"colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.05, 1.0),
        #"min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
    }
    kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

    drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
    added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
    oof = np.zeros(len(train))
    preds = np.zeros(len(test))
    features = [col for col in train.columns if col not in config.col_ignore]
    cats = [col for col in features if col not in config.num_cols]
    cats.extend(added_fe)
    m = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

        x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
        y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
        x_test = test[features].copy()

        TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


        # adding extra features

        x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
        x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
        x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

        x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
        x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
        x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

        x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
        x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
        x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')



        for col in added_fe:
          TE.fit(x_train[col], y_train)
          x_train[f'{col}_TE'] = TE.transform(x_train[col])
          x_val[f'{col}_TE'] = TE.transform(x_val[col])
          x_test[f'{col}_TE'] = TE.transform(x_test[col])

        for col in features:
            TE.fit(x_train[col], y_train)
            x_train[f'{col}_TE'] = TE.transform(x_train[col])
            x_val[f'{col}_TE'] = TE.transform(x_val[col])
            x_test[f'{col}_TE'] = TE.transform(x_test[col])

        x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
        x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
        x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

        x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
        x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
        x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

        x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
        x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
        x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

        x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
        x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
        x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

        for cat in cats:
            x_train[cat] =  x_train[cat].fillna("MISSING")
            x_val[cat] = x_val[cat].fillna("MISSING")
            x_test[cat] = x_test[cat].fillna("MISSING")
            x_train[cat] =  x_train[cat].astype('str')
            x_val[cat] = x_val[cat].astype('str')
            x_test[cat] = x_test[cat].astype('str')

        print(x_train.columns)

        model = CatBoostRegressor(**params,
                                  task_type = "GPU",
                                  grow_policy = 'Lossguide',
                                  random_state = 42,
                                  cat_features = cats,
                                  verbose = 250,
                                  loss_function='RMSE')

        model.fit(x_train, y_train)

        val_preds = model.predict(x_val)

        oof[test_idx] = val_preds

        preds += model.predict(x_test)/config.n_splits

        score = rmse(y_val, val_preds)

        m.append(score)

        print(f'Fold: {fold+1}, Score: {score}')

    print(f"The average CV is {np.average(m)}")
    return np.average(m)

In [ ]:

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)

[I 2025-02-13 07:56:38,228] A new study created in memory with name: no-name-1ad79710-4c21-4e5e-902d-0b36920d12b6


Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870612	total: 24.4ms	remaining: 24.4s
250:	learn: 38.6235954	total: 3.54s	remaining: 10.6s
500:	learn: 38.5955037	total: 6.91s	remaining: 6.88s
750:	learn: 38.5713710	total: 10.2s	remaining: 3.39s
999:	learn: 38.5488252	total: 13.4s	remaining: 0us
Fold: 1, Score: 38.63831541353275
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weigh

[I 2025-02-13 08:21:16,928] Trial 0 finished with value: 38.655052079763585 and parameters: {'l2_leaf_reg': 14}. Best is trial 0 with value: 38.655052079763585.


Fold: 25, Score: 38.74451965980291
The average CV is 38.655052079763585
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870526	total: 19.6ms	remaining: 19.6s
250:	learn: 38.6239220	total: 3.53s	remaining: 10.5s
500:	learn: 38.5957718	total: 6.83s	remaining: 6.8s
750:	learn: 38.5704951	total: 10s	remaining: 3.33s
999:	learn: 38.5477271	total: 13.3s	remaining: 0us
Fold: 1, Score: 38.639512150313664
Index(['Brand', 'Material', 'Size', 'Compartments

[I 2025-02-13 08:45:52,166] Trial 1 finished with value: 38.65481370548474 and parameters: {'l2_leaf_reg': 9}. Best is trial 1 with value: 38.65481370548474.


...

Fold: 25, Score: 38.745140740862574
The average CV is 38.65497525470145
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870663	total: 19.8ms	remaining: 19.8s
250:	learn: 38.6239946	total: 3.6s	remaining: 10.7s
500:	learn: 38.5968771	total: 6.88s	remaining: 6.86s
750:	learn: 38.5721031	total: 10.1s	remaining: 3.34s
999:	learn: 38.5498348	total: 13.2s	remaining: 0us
Fold: 1, Score: 38.63782143046331
Index(['Brand', 'Material', 'Size', 'Compartment

[I 2025-02-13 09:35:09,718] Trial 3 finished with value: 38.655598864166215 and parameters: {'l2_leaf_reg': 15}. Best is trial 1 with value: 38.65481370548474.


Fold: 25, Score: 38.74571921447004
The average CV is 38.655598864166215
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870663	total: 19.7ms	remaining: 19.7s
250:	learn: 38.6240016	total: 3.54s	remaining: 10.6s
500:	learn: 38.5966107	total: 6.85s	remaining: 6.82s
750:	learn: 38.5723887	total: 10.1s	remaining: 3.34s
999:	learn: 38.5497621	total: 13.2s	remaining: 0us
Fold: 1, Score: 38.638115164826395
Index(['Brand', 'Material', 'Size', 'Compartme

[I 2025-02-13 09:59:47,532] Trial 4 finished with value: 38.655336749219146 and parameters: {'l2_leaf_reg': 15}. Best is trial 1 with value: 38.65481370548474.


Fold: 25, Score: 38.745050604689986
The average CV is 38.655336749219146


In [ ]:
print('Best hyperparameters:', study.best_params)
print('Best RMSE:', study.best_value)

Best hyperparameters: {'l2_leaf_reg': 9}
Best RMSE: 38.65481370548474


In [ ]:
from google.colab import runtime
runtime.unassign()

In [17]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = preds
submission.to_csv("submission.csv", index = False)

submission

,id,Price
0,300000,81.465299
1,300001,82.811307
2,300002,88.803254
3,300003,78.794630
4,300004,79.203256
...,...,...
199995,499995,81.908618
199996,499996,72.506764
199997,499997,82.965906
199998,499998,82.152603


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=model.get_feature_importance(), y=x_train.columns)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

In [18]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission with 20 kfold TE combo'

100% 4.74M/4.74M [00:00<00:00, 8.57MB/s]
Successfully submitted to Backpack Prediction Challenge

In [ ]:
!kaggle competitions submissions -c {competition}